# Rust 114: Smart Pointers and Interior Mutability — Practice

Companion exercises for `14_smart-pointers-and-interior-mutability_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its eleven demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. `Box`: The Heap Pointer](#1-box-the-heap-pointer)
- [2. `Deref` and `DerefMut`: The Coercion Machine](#2-deref-and-derefmut-the-coercion-machine)
- [3. `Drop`: The Destructor](#3-drop-the-destructor)
- [4. `Rc`: Many Owners, One Allocation](#4-rc-many-owners-one-allocation)
- [5. `Cell`: Copy Into and Copy Out](#5-cell-copy-into-and-copy-out)
- [6. `RefCell`: The Runtime Borrow Check](#6-refcell-the-runtime-borrow-check)
- [7. `Rc<RefCell<T>>`: Sharing and Mutating](#7-rcrefcellt-sharing-and-mutating)
- [8. `OnceCell` and `OnceLock`: Write Once](#8-oncecell-and-oncelock-write-once)
- [9. `Arc` and the Thread Boundary](#9-arc-and-the-thread-boundary)
- [10. Cycles and `Weak`: The Leak](#10-cycles-and-weak-the-leak)
- [11. Reading the Compiler: E-Codes](#11-reading-the-compiler-e-codes)
- [14. Review: Mixed Problems](#14-review-mixed-problems)

§1–§11 each have one exercise for every demonstration cell of the example notebook (the Contents lists only the exercise-bearing sections; §12's receipts ledger and §13's cheat sheet are reading). When a check fails, the fastest route back is §13 of the example notebook, which is the cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or impl may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name does the import, and later cells rely on persistence — a repeated `use` is a redefinition (E0252).

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Three notes specific to this chapter

- **Given code is complete and stays.** The `Chip`/`Rack` trait scaffolding (§3.1), `Fork`'s counts (§4.1), `Thermometer`'s readings (§6.1), `Flock`'s tree (§10.1) and the `Critter` cycle (§10.2) are declared for you — solve only the `TODO` bodies.
- **§10.2's exercise is a reading, not a coding task.** The cycle it exhibits leaks *by design*: answer the objectives by reading the cell, then run it once to see the silence. There is no check line — the absence of `[drop]` lines *is* the check.
- **The kernel keeps end-of-cell bindings alive**, so §3.1's drop evidence comes from an explicit inner scope and `mem::drop` — not from "when the cell ends". §0 of the example notebook has the honest caveat.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- `never used`/`never read`/`never constructed` — stub-induced dead code: whole given structs and methods sit unused until your body calls them.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 of the example notebook — and its §11 when a diagnostic surprises you.


## 1. `Box`: The Heap Pointer

### Exercise 1.1 — The recursion wall, and the payload behind it

**Goal**: build the recursive `Course` type the example notebook's §1 made legal with `Box`, walk it with a recursive fn, and move a payload *out* of a `Box` — the two moves E0072's fix unlocks.

**Given**: the `Course` enum is declared exactly as the diagnostic proposes — the recursive tail behind a `Box`.

**Objectives**

- implement `course_length` to walk the chain and count courses — recursion through the `Box` tail, `0` at `Terminus`
- implement `unbox_name` to move the `String` out of the box and return it — the deref-move only `Box` allows
- keep the given enum and the signatures unchanged

**See**: §1.1 of the example notebook.

**Hint**: the walk is §1.1's `menu_sum` with counting instead of summing — `match` on the variant, recurse through `&course.tail`, return `0` at `Terminus`. For the move-out: `*boxed` consumes the `Box` and hands you the payload.


In [ ]:
enum Course {
    Serving(u32, Box<Course>), // the Box keeps the size finite (E0072)
    Terminus,
}

/// Walk the chain and count the servings - recursion through the Box tail.
///
/// TODO: replace `todo!()` with the recursive match.
fn course_length(course: &Course) -> usize {
    todo!()
}

/// Move the name String out of the box - the deref-move only Box allows.
///
/// TODO: replace `todo!()` with the deref-move.
fn unbox_name(boxed: Box<String>) -> String {
    todo!()
}

// --- check (do not modify) ---
let menu = Course::Serving(1, Box::new(Course::Serving(
    2, Box::new(Course::Terminus))));
assert_eq!(course_length(&menu), 2);
assert_eq!(course_length(&Course::Terminus), 0);
assert_eq!(unbox_name(Box::new("liberated".to_string())), "liberated");
assert_eq!(std::mem::size_of::<Course>(), 16); // handle-sized, not infinite
println!("Exercise 1.1 passed");


## 2. `Deref` and `DerefMut`: The Coercion Machine

### Exercise 2.1 — A hand-rolled pointer, coercions included

**Goal**: build the two-trait pointer the example notebook's §2.1 built — `Deref` for the read side, `DerefMut` for the write side — and let the compiler's coercion ladder do the rest.

**Given**: the `Safe` newtype is declared with its constructor; the impl blocks are the exercise.

**Objectives**

- implement `Deref` for `Safe` — the `type Target = Inner;` line is **given** (E0046 without it); the `deref` borrow of the payload is yours
- implement `DerefMut` for `Safe` — the mutable borrow of the payload
- implement `safe_launch` to arm the payload through `DerefMut` and return its name — two `&mut Inner` methods, no `&mut` anywhere in your signature
- keep the given struct and signatures unchanged

**See**: §2.1 of the example notebook.

**Hint**: the impls are three lines each: `type Target = Inner;` then `&self.payload` / `&mut self.payload`. After that, `armed.launch()` resolves through the ladder, and `&safe` coerces to `&str` in the check's `as_str()` call.


In [ ]:
use std::ops::{Deref, DerefMut};

pub struct Inner {
    armed: bool,
    name: String,
}
impl Inner {
    pub fn arm(&mut self) {
        self.armed = true;
    }
    pub fn launch(&self) -> String {
        if self.armed {
            format!("{} away", self.name)
        } else {
            format!("{} on the pad", self.name)
        }
    }
    pub fn name(&self) -> &str {
        &self.name
    }
}

pub struct Safe {
    payload: Inner,
}
impl Safe {
    pub fn new(name: &str) -> Self {
        Safe { payload: Inner { armed: false, name: name.to_string() } }
    }
}

// The Target line is given (E0046 without it); the borrow is yours.
impl Deref for Safe {
    type Target = Inner;
    fn deref(&self) -> &Self::Target {
        todo!()
    }
}

// TODO: implement DerefMut for Safe - deref_mut borrows the payload mutably.
impl DerefMut for Safe {
    fn deref_mut(&mut self) -> &mut Self::Target {
        todo!()
    }
}

/// Arm through DerefMut, then read through Deref - no &mut in this signature.
///
/// TODO: replace `todo!()` with: rebind into a `mut` local, arm the payload,
/// then return its launch().
fn safe_launch(rocket: Safe) -> String {
    todo!()
}

// --- check (do not modify) ---
let cold = Safe::new("cold");
assert_eq!(cold.launch(), "cold on the pad"); // method via Deref
assert_eq!(cold.name(), "cold"); // &Safe -> &Inner -> &str in one return
let hot = Safe::new("hot");
assert_eq!(safe_launch(hot), "hot away"); // arm() went through DerefMut
println!("Exercise 2.1 passed");


## 3. `Drop`: The Destructor

### Exercise 3.1 — The last word, in the right order

**Goal**: give a type its destructor and predict the drop order the example notebook's §3 proved — then make the evidence print inside one cell with an inner scope and `mem::drop`.

**Given**: the `Chip` trait and its `Rack` wrapper are complete — `Rack` owns a `Box<dyn Chip>`, and the dyn dispatch (chapter 10) prints through your `Drop` impls.

**Objectives**

- implement `Drop` for both `Fpga` and `Asic` — each prints `[drop] <its name>` via the `name()` the trait provides
- implement `Rack::tear_down` to drop the boxed chip *immediately* — the sanctioned early exit, `std::mem::drop`
- implement `stack_and_pop` to return the order the drops print in: push three labels into a `Vec`, drop the *middle* one early, and return the drop order as a `&'static str` — LIFO at the scope's close
- keep the given trait, wrapper and signatures unchanged

**See**: §3.1 of the example notebook.

**Hint**: the destructor is `impl Drop for Fpga { fn drop(&mut self) { println!("[drop] {}", self.name()); } }` — `&mut self`, no arguments. For `stack_and_pop`: the drops at the closing brace run last-pushed-first, and the early `mem::drop` lands exactly where you call it.


In [ ]:
trait Chip {
    fn name(&self) -> &'static str;
}
struct Fpga;
impl Chip for Fpga {
    fn name(&self) -> &'static str {
        "fpga"
    }
}
struct Asic;
impl Chip for Asic {
    fn name(&self) -> &'static str {
        "asic"
    }
}
struct Rack {
    slot: Box<dyn Chip>,
}
impl Rack {
    fn load(chip: Box<dyn Chip>) -> Self {
        Rack { slot: chip }
    }
    /// Drop the boxed chip immediately - the sanctioned early exit.
    ///
    /// TODO: replace `todo!()` with the mem::drop call.
    fn tear_down(self) {
        todo!()
    }
}

/// Push three chips, pop the middle one early, report the drop order.
///
/// TODO: replace `todo!()` with the LIFO probe - inner scope, mem::drop,
/// and return "middle, top, bottom" once the drops prove it.
fn stack_and_pop() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
Rack::load(Box::new(Asic)).tear_down(); // prints [drop] asic, immediately
assert_eq!(stack_and_pop(), "middle, top, bottom");
println!("Exercise 3.1 passed");


## 4. `Rc`: Many Owners, One Allocation

### Exercise 4.1 — The count is the contract

**Goal**: drive `Fork`'s shared payload through the count's whole life — clone up, refuse mutation while shared, unwrap the last handle — the exact dance §4.1 proved on the example notebook's `Rc`.

**Given**: the `Fork` struct and its constructor are complete; the payload is a plain `u32` so every count change is visible.

**Objectives**

- implement `shared_tag` to return `(strong_count, is_same_allocation, can_get_mut)` for a `Fork` with **two** handles — `Rc::strong_count`, `Rc::ptr_eq` and `Rc::get_mut` are the three probes
- implement `last_handle` to steal the payload with `Rc::try_unwrap` when (and only when) the count is one — returning `Err`'s `Rc` back as `None`'s partner is the refusal half
- keep the given struct and signatures unchanged

**See**: §4.1 of the example notebook.

**Hint**: `get_mut(&mut r.clone())` is the honest shared-state probe — clone, ask, drop the clone. `try_unwrap` returns `Result<T, Rc<T>>`: the `Err` hands the handle back, which is why nothing leaks when the count is not one.


In [ ]:
use std::rc::Rc;

struct Fork {
    tine: u32,
}
impl Fork {
    fn new(tine: u32) -> Self {
        Fork { tine }
    }
}

/// Probe a two-handle Fork: (strong count, same allocation, get_mut ok?).
///
/// TODO: replace `todo!()` with the three probes against a cloned handle.
fn shared_tag(fork: &Rc<Fork>) -> (usize, bool, bool) {
    todo!()
}

/// Steal the payload if this is the last handle; None while it is shared.
///
/// TODO: replace `todo!()` with the try_unwrap + map.
fn last_handle(handle: Rc<Fork>) -> Option<u32> {
    todo!()
}

// --- check (do not modify) ---
let fork = Rc::new(Fork::new(4));
assert_eq!(shared_tag(&fork), (2, true, false)); // probe's clone live: count 2
assert_eq!(Rc::strong_count(&fork), 1); // the probe's clone is gone again
let solo = Rc::new(Fork::new(7));
assert_eq!(last_handle(solo), Some(7)); // count 1: the steal succeeds
assert_eq!(last_handle(Rc::clone(&fork)), None); // count 2: refused
assert_eq!(Rc::strong_count(&fork), 1); // nothing leaked in the refusal
println!("Exercise 4.1 passed");


## 5. `Cell`: Copy Into and Copy Out

### Exercise 5.1 — The ch12 counter, from the outside in

**Goal**: rebuild the laziness counter chapter 12 smuggled and chapter 14 explained — mutation through a shared reference, with `Cell`'s three moves (`set`, `get`, `take`) each earning their line.

**Given**: nothing — `Cell` needs no scaffolding; that *is* the lesson.

**Objectives**

- implement `tick_n` to add `n` to a shared `Cell<usize>` **through a shared borrow** — the closure-friendly mutation `&T` alone can never do
- implement `drain_ticks` to return the counter's total and reset it to zero in one move — `take()` puts `T::default()` in and hands the old value out
- keep the signatures unchanged; the check captures the counter in a closure exactly as ch12's laziness proof did

**See**: §5.1 of the example notebook.

**Hint**: `counter.set(counter.get() + n)` is the whole first body — the read and the write both go through copies, which is why no borrow is ever handed out. `take()` is `T::default()`-in, old-value-out.


In [ ]:
use std::cell::Cell;

/// Add n to the counter through a SHARED borrow - Cell's whole trick.
///
/// TODO: replace `todo!()` with the get + set body.
fn tick_n(counter: &Cell<usize>, n: usize) {
    todo!()
}

/// Return the total and reset the counter to zero in one move.
///
/// TODO: replace `todo!()` with the take call.
fn drain_ticks(counter: &Cell<usize>) -> usize {
    todo!()
}

// --- check (do not modify) ---
let counter = Cell::new(0);
let capture = |c: &Cell<usize>| tick_n(c, 3); // a closure captures &Cell:
capture(&counter); // mutation through a shared borrow, compile-checked
capture(&counter);
assert_eq!(counter.get(), 6);
assert_eq!(drain_ticks(&counter), 6);
assert_eq!(counter.get(), 0); // drained to T::default()
assert_eq!(std::mem::size_of::<Cell<usize>>(), 8); // zero overhead over usize
println!("Exercise 5.1 passed");


## 6. `RefCell`: The Runtime Borrow Check

### Exercise 6.1 — The loan office, robbed on camera

**Goal**: run `Thermometer`'s loan office the way §6.1 did — take the shared loans, then trip the flag and *catch the panic*, pinning the message **and** the payload type (chapter 11's habit misses it: this std panic boxes a `String`).

**Given**: the `Thermometer` struct and its free `read_both` fn (a borrower from outside the type) are complete; the loan methods are the exercise.

**Objectives**

- implement `borrow_both` to return the two readings under one shared loan — `borrow()` twice is legal, the guard derefs
- implement `bump` to take an exclusive loan and add the offset — `borrow_mut()`, one statement, the guard drops at the semicolon
- implement `violation` to trip the flag — a `borrow_mut` while `read_both`'s loan is live — catch it with `catch_unwind`, and return the payload **as a `String`**
- keep the given struct, fn and signatures unchanged

**See**: §6.1 of the example notebook.

**Hint**: the payload is `String`, not `&str` — `downcast_ref::<&str>()` returns `None`; the `String` arm is the one that catches `"RefCell already borrowed"`. The closure needs `AssertUnwindSafe` because it borrows the thermometer.


In [ ]:
use std::cell::RefCell;

struct Thermometer {
    wet: f64,
    dry: f64,
}
impl Thermometer {
    fn new(wet: f64, dry: f64) -> Self {
        Thermometer { wet, dry }
    }
}

/// Both readings under one shared loan - two borrows coexist.
///
/// TODO: replace `todo!()` with the two borrow() reads.
fn borrow_both(t: &RefCell<Thermometer>) -> (f64, f64) {
    todo!()
}

/// Exclusive loan through a shared binding - the guard lives one line.
///
/// TODO: replace `todo!()` with the borrow_mut + add.
fn bump(t: &RefCell<Thermometer>, by: f64) {
    todo!()
}

/// A borrower from outside the type: the loan is held while this runs.
fn read_both(t: &RefCell<Thermometer>) -> (f64, f64) {
    let guard = t.borrow();
    (guard.wet, guard.dry)
}

/// Trip the flag, catch the panic, return the payload as a String.
///
/// TODO: replace `todo!()` with: borrow the cell, then catch_unwind a
/// calibrate; downcast the payload as a String.
fn violation(cell: &RefCell<Thermometer>) -> String {
    todo!()
}

// --- check (do not modify) ---
let t = RefCell::new(Thermometer::new(18.5, 22.0));
assert_eq!(borrow_both(&t), (18.5, 22.0));
assert_eq!(read_both(&t), (18.5, 22.0)); // a second borrower, same cell
bump(&t, 0.5); // exclusive loan, one statement long
assert_eq!(borrow_both(&t), (19.0, 22.0)); // only the WET bulb moved
assert_eq!(violation(&t), "RefCell already borrowed");
println!("Exercise 6.1 passed");


## 7. `Rc<RefCell<T>>`: Sharing and Mutating

### Exercise 7.1 — One flag, three owners

**Goal**: rebuild §7.1's ledger with `Cargo` — three owners, writes through two of them, and the proof that the borrow flag lives in the **`RefCell`**, not in any handle: a violation *across two different clones* still panics.

**Given**: the `Cargo` struct and its constructor are complete.

**Objectives**

- implement `record` to push a reading through *your* handle — `borrow_mut().push(..)`, the temporary-guard idiom
- implement `snapshot` to return a copy of the log under a shared loan — `borrow()` + `.clone()` on the `Vec`
- implement `cross_handle_violation` to hold an exclusive loan on the *first* clone and take a shared loan on the *second* — catching the panic proves the flag is per-`RefCell`, not per-handle
- keep the given struct and signatures unchanged

**See**: §7.1 of the example notebook.

**Hint**: `Rc::clone(&cell)` for each owner; `borrow()` is refused while *any* handle's `borrow_mut` guard lives. The payload, again, is a `String`.


In [ ]:
// (RefCell is already imported - a use line lives for the notebook.)
struct Cargo {
    log: Vec<u32>,
}
impl Cargo {
    fn new(first: u32) -> Self {
        Cargo { log: vec![first] }
    }
}

/// Push a reading through this handle - the temporary-guard idiom.
///
/// TODO: replace `todo!()` with the borrow_mut + push.
fn record(cell: &std::rc::Rc<RefCell<Cargo>>, reading: u32) {
    todo!()
}

/// A copy of the log under a shared loan.
///
/// TODO: replace `todo!()` with the borrow + clone.
fn snapshot(cell: &std::rc::Rc<RefCell<Cargo>>) -> Vec<u32> {
    todo!()
}

/// Two owners, one flag: exclusive on one clone, shared on the other.
///
/// TODO: replace `todo!()` with: catch_unwind a closure that holds
/// borrow_mut on `held` while borrowing `other`; return whether it panicked.
fn cross_handle_violation(held: &std::rc::Rc<RefCell<Cargo>>, other: &std::rc::Rc<RefCell<Cargo>>) -> bool {
    todo!()
}

// --- check (do not modify) ---
let cell = std::rc::Rc::new(RefCell::new(Cargo::new(100)));
let writer = cell.clone(); // Rc::clone: same allocation, strong count 2
let reader = cell.clone(); // and 3
record(&writer, 250);
record(&reader, 400);
assert_eq!(snapshot(&cell), vec![100, 250, 400]);
assert!(cross_handle_violation(&writer, &reader)); // one flag, three owners
assert_eq!(std::rc::Rc::strong_count(&cell), 3);
println!("Exercise 7.1 passed");


## 8. `OnceCell` and `OnceLock`: Write Once

### Exercise 8.1 — First writer wins, forever

**Goal**: rebuild §8.1's config cell — the first `get_or_init` pays, every later call reads, and an overwrite attempt is an `Err`, not a panic.

**Given**: nothing — the API is the exercise.

**Objectives**

- implement `provision` to initialise the shared `OnceCell<String>` with its first writer's stamp, or return the existing value untouched — `get_or_init` runs your closure only when the cell is empty
- implement `try_overwrite` to attempt a `set` on a filled cell and return the refusal — `Err(the_value_you_tried_to_set)`
- keep the signatures unchanged; the check's two callers with different stamps prove only the first closure ever runs

**See**: §8.1 of the example notebook.

**Hint**: `get_or_init(|| format!("{who}-built-this"))` returns `&String` — deref-copied out for the returned `String`. `set` returns `Result<(), String>`: the `Err` carries exactly what you tried to put in.


In [ ]:
use std::cell::OnceCell;

/// Initialise the cell with this caller's stamp, or read the existing value.
///
/// TODO: replace `todo!()` with the get_or_init + to_owned.
fn provision(cell: &OnceCell<String>, who: &str) -> String {
    todo!()
}

/// Attempt an overwrite on a filled cell; return the refusal's payload.
///
/// TODO: replace `todo!()` with the set + expect_err.
fn try_overwrite(cell: &OnceCell<String>, value: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
let config: OnceCell<String> = OnceCell::new();
assert_eq!(provision(&config, "alpha"), "alpha-built-this");
assert_eq!(provision(&config, "beta"), "alpha-built-this"); // closure skipped
assert_eq!(try_overwrite(&config, "mutiny"), "mutiny"); // refused, returned
assert_eq!(config.get().map(|s| s.as_str()), Some("alpha-built-this"));
assert_eq!(std::mem::size_of::<OnceCell<u32>>(), 8); // one discriminant wide
println!("Exercise 8.1 passed");


## 9. `Arc` and the Thread Boundary

### Exercise 9.1 — Cross the boundary, legally

**Goal**: do §9.1's thread walk for yourself — move an `Arc` into a spawned thread, fan four workers over one allocation, and feel `join` hand the handles back (the strong count falling is the receipt).

**Given**: nothing but the lesson: `Rc` cannot cross; `Arc` can; both are read-only through the shared handle.

**Objectives**

- implement `sum_parallel` to fan `workers` threads over the same `Arc<Vec<u64>>` — each computes its share's sum via `chunk` arithmetic, `join()` collects, and the total returns
- implement `try_send_rc` to demonstrate the refusal at the type level *without* spawning: return whether `Rc<i32>` is usable in the closure's place — spelled as a `false` with the E0277 text as the doc comment
- keep the signatures unchanged

**See**: §9.1 of the example notebook.

**Hint**: split the slice first: `let chunk = data.len() / workers;` then each thread's move-closure owns an `Arc::clone` and indexes its band. `join()` returns the closure's value — `handle.join().expect("worker panicked")`.


In [ ]:
use std::sync::Arc;
use std::thread;

/// Fan `workers` threads over one Arc allocation and sum the whole vector.
///
/// TODO: replace `todo!()` with the clone-per-worker loop + join sum.
fn sum_parallel(data: Arc<Vec<u64>>, workers: usize) -> u64 {
    todo!()
}

/// The Rc refusal, as a function: `Rc<i32>` may not cross a thread boundary.
///
/// TODO: replace `todo!()` with `false` - and read the diagnostic this
/// comment captures (E0277, from the example notebook's section 9):
///   error[E0277]: `Rc<i32>` cannot be sent between threads safely
///   = note: required for `Rc<i32>` to implement `Send`
fn try_send_rc() -> bool {
    todo!()
}

// --- check (do not modify) ---
let data = Arc::new(vec![1, 2, 3, 4, 5, 6, 7, 8]);
assert_eq!(sum_parallel(Arc::clone(&data), 4), 36); // 4 bands of 2
assert_eq!(sum_parallel(Arc::clone(&data), 1), 36); // one band of 8
assert_eq!(Arc::strong_count(&data), 1); // every worker handle dropped at join
assert!(!try_send_rc());
println!("Exercise 9.1 passed");


## 10. Cycles and `Weak`: The Leak

### Exercise 10.1 — The back-edge that does not own

**Goal**: wire §10.1's family tree — ownership flows down through `Rc`, the parent pointer flows back through `Weak`, and the `[drop]` lines inside the cell prove no leak.

**Given**: the `Gosling` node is declared with its `Drop` printer — `parent` is a `Weak`, `children` are strong.

**Objectives**

- implement `Gosling::adopt` to attach a child: the child's `parent` becomes a downgrade of the parent, the parent's `children` gains a strong clone of the child — both directions, one fn
- implement `Gosling::climb` to take one `upgrade()` step up the `Weak` edge — `None` at an orphan (nothing behind it)
- keep the given struct, `Drop` impl and signatures unchanged; the check's inner scope makes both `[drop]` lines print *inside the cell* — the no-leak proof §10.1 made

**See**: §10.1 of the example notebook.

**Hint**: `*child.parent.borrow_mut() = Rc::downgrade(parent);` is the back-edge — it bumps weak_count, not strong_count. The climb is one `self.parent.borrow().upgrade()` — the caller loops if they want the top.


In [ ]:
// (RefCell and Rc are already imported; Weak joins the notebook here.)
use std::rc::Weak;

struct Gosling {
    name: &'static str,
    parent: RefCell<Weak<Gosling>>, // the back-edge: watched, not owned
    children: RefCell<Vec<Rc<Gosling>>>, // ownership flows down only
}
impl Gosling {
    fn orphan(name: &'static str) -> Rc<Gosling> {
        Rc::new(Gosling {
            name,
            parent: RefCell::new(Weak::new()),
            children: RefCell::new(vec![]),
        })
    }
    /// Attach `child`: its Weak parent becomes us, our children gain it.
    ///
    /// TODO: replace `todo!()` with the downgrade + the strong push.
    fn adopt(self: &Rc<Gosling>, child: &Rc<Gosling>) {
        todo!()
    }
    /// Climb toward the root; Some(ancestor) or None when already at the top.
    ///
    /// TODO: replace `todo!()` with one upgrade step up the Weak edge.
    fn climb(self: &Rc<Gosling>) -> Option<Rc<Gosling>> {
        todo!()
    }
}
impl Drop for Gosling {
    fn drop(&mut self) {
        println!("[drop] Gosling {}", self.name);
    }
}

// --- check (do not modify) ---
let answer = {
    let web = Gosling::orphan("web");
    let spoke = Gosling::orphan("spoke");
    web.adopt(&spoke);
    assert_eq!(std::rc::Rc::strong_count(&web), 1); // the back-edge never counts
    assert_eq!(std::rc::Rc::weak_count(&web), 1);
    let up = spoke.climb().expect("spoke has a parent");
    assert_eq!(up.name, "web"); // one upgrade step reached the root
    assert!(web.climb().is_none()); // the orphan has nothing above it
    "spoke->web"
}; // brace: both [drop] lines print here - no leak
assert_eq!(answer, "spoke->web");
println!("Exercise 10.1 passed");


### Exercise 10.2 — The leak, exhibited (a reading)

**Goal**: read the cycle the example notebook's §10 proved and answer *before running*: which `[drop]` lines will this cell print?

**Given**: the `Critter` pair below forms a strong cycle — `a` owns `b`, `b` owns `a` — each with a `Drop` printer. **This is a reading exercise**: the leak is the demonstration. There is no check line; the silence *is* the check.

**Objectives**

- predict: how many `[drop]` lines print when the scope ends, and why the counts cannot reach zero
- run the cell once and confirm the silence
- name the one-word type change (`Rc` → what?) that would let the drops run

**See**: §10 of the example notebook.

**Hint**: both strong counts rest at 2 inside the scope (one handle + one cycle edge). At the scope's close each side loses its handle — leaving count 1 from the other side of the cycle, forever. The fix is the back-edge: `Weak<Critter>`.


In [ ]:
// (RefCell and Rc are already imported - persistence, not re-import.)

struct Critter {
    name: &'static str,
    friend: RefCell<Option<Rc<Critter>>>, // strong in BOTH directions: a cycle
}
impl Critter {
    fn new(name: &'static str) -> Rc<Critter> {
        Rc::new(Critter { name, friend: RefCell::new(None) })
    }
}
impl Drop for Critter {
    fn drop(&mut self) {
        println!("[drop] Critter {}", self.name); // never prints below
    }
}
{
    let a = Critter::new("a");
    let b = Critter::new("b");
    *a.friend.borrow_mut() = Some(Rc::clone(&b));
    *b.friend.borrow_mut() = Some(Rc::clone(&a));
    println!("counts inside the scope: a={}, b={}",
        std::rc::Rc::strong_count(&a), std::rc::Rc::strong_count(&b));
}
// The scope ended. Predicted [drop] lines: NONE - both counts froze at 1,
// held by the cycle's far end. Swap friend to Weak<Critter> and both print.
println!("Exercise 10.2 passed (the silence was the check)");


## 11. Reading the Compiler: E-Codes

### Exercise 11.1 — The catalogue, dodged live

**Goal**: walk the chapter's diagnostics with working callsites — every dodge in one cell, each forbidden shape kept as its captured comment, exactly as §11.1 of the example notebook does.

**Given**: nothing but the module scaffolding below; each doc comment names the diagnostic its stub side-steps.

**Objectives**

- implement `box_tail` to make the recursive call legal — the E0072 dodge, one `Box` at the recursive position
- implement `local_drop` to report the payload of a type that owns its destructor — the E0117/E0120 dodge: `Drop` only on a local struct, never on `i32`
- implement `thread_safe_choice` to return which pointer may cross a thread boundary — the E0277 dodge, `Arc` over `Rc`
- keep the signatures unchanged

**See**: §11.1 of the example notebook.

**Hint**: each body is one line plus a comment carrying the diagnostic you are dodging — the same discipline as the example notebook's catalogue cell.


In [ ]:
// E0072 dodged: the recursive position carries a Box, so the size is finite.
enum Strand {
    Base(u8, Box<Strand>),
    Stop,
}

/// The E0072 dodge: Box at the recursive tail.
///
/// TODO: replace `todo!()` with the count of bases before Stop.
fn box_tail(strand: &Strand) -> usize {
    todo!()
}

/// The E0117/E0120 dodge: Drop only on a LOCAL type (this one), never i32.
///
/// TODO: replace `todo!()` with the payload's value, read through the guard.
fn local_drop(guard: &Sensor) -> u32 {
    todo!()
}
struct Sensor {
    reading: u32,
}
impl Drop for Sensor {
    fn drop(&mut self) {} // legal: Sensor is local; impl Drop for i32 is not
}

/// The E0277 dodge: the pointer that may cross a thread boundary.
///
/// TODO: replace `todo!()` with true for the Arc spelling, false for Rc.
fn thread_safe_choice(use_arc: bool) -> bool {
    todo!()
}

// --- check (do not modify) ---
let strand = Strand::Base(1, Box::new(Strand::Base(2, Box::new(Strand::Stop))));
assert_eq!(box_tail(&strand), 2);
let s = Sensor { reading: 42 };
assert_eq!(local_drop(&s), 42);
assert!(thread_safe_choice(true));
assert!(!thread_safe_choice(false));
println!("Exercise 11.1 passed");


## 14. Review: Mixed Problems

Three problems, each assembling several sections' worth of machinery: a registry that shares and mutates, a once-initialised lattice with weak back-edges, and a destructor-timing report — every earlier exercise's move, used once more together.

### Exercise 14.1 — The registry: share it, then mutate it

**Goal**: build the canonical `Rc<RefCell<T>>` service — many named holders, one log, every write and read through the shared handle — then hand a read-only snapshot to a latecomer.

**Given**: the `Registry` struct and its constructor are complete.

**Objectives**

- implement `Registry::enroll` to add one name — its `&mut self` signature is the given hint: the *caller* supplies the exclusivity through `borrow_mut()`
- implement `Registry::roll_call` to return the sorted names under a shared loan — the `borrow()` half
- implement `enroll_all` to apply `enroll` through a *cloned* handle for every name in order — `Rc::clone`, then `borrow_mut()` per name, the temporary-guard idiom at the call site
- implement `latecomer_view` to return the roll call through a fresh clone taken *after* the writes — proving the latecomer sees the whole log (the clone drops inside, which is the receipt itself)
- keep the given struct and signatures unchanged

**See**: §7.1, §4.1 and §5.1 of the example notebook.

**Hint**: the registry lives in `Rc<RefCell<Registry>>` — `enroll` needs `borrow_mut()`, `roll_call` only `borrow()`, and every method call runs through the one runtime flag, however many handles hold it.


In [ ]:
// (RefCell and Rc are already imported.)

struct Registry {
    names: Vec<String>,
}
impl Registry {
    fn new() -> Self {
        Registry { names: vec![] }
    }
    /// Add one name. The &mut self says: call me through borrow_mut().
    ///
    /// TODO: replace `todo!()` with the push.
    fn enroll(&mut self, name: &str) {
        todo!()
    }
    /// The names, sorted, under a shared loan.
    ///
    /// TODO: replace `todo!()` with the borrow + sort + clone.
    fn roll_call(&self) -> Vec<String> {
        todo!()
    }
}

/// Enroll every name through a cloned handle, in order.
///
/// TODO: replace `todo!()` with the Rc::clone + loop over enroll.
fn enroll_all(registry: &Rc<RefCell<Registry>>, names: &[&str]) {
    todo!()
}

/// The latecomer's view: a fresh clone AFTER the writes sees the whole log.
///
/// TODO: replace `todo!()` with the roll_call through a fresh Rc::clone.
fn latecomer_view(registry: &Rc<RefCell<Registry>>) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let registry = Rc::new(RefCell::new(Registry::new()));
enroll_all(&registry, &["delta", "alpha", "charlie"]);
registry.borrow_mut().enroll("bravo"); // a fourth holder, one line
assert_eq!(latecomer_view(&registry),
    vec!["alpha", "bravo", "charlie", "delta"]);
assert_eq!(Rc::strong_count(&registry), 1); // the latecomer's clone dropped
println!("Exercise 14.1 passed");


### Exercise 14.2 — The lattice: once-built, weakly linked

**Goal**: combine §8 and §10 — a node cache behind `OnceCell` initialisation, with parent links as `Weak` so the whole lattice drops when the strong handles drop.

**Given**: the `Brick` node is declared; the cache type alias is spelled for you.

**Objectives**

- implement `Brick::link` to record a parent as a `Weak` back-edge — the §10.1 move
- implement `Brick::parent_name` to read through the `Weak` — `None` for an orphan
- implement `build_cache` to initialise the `OnceCell` cache with the *first* caller's node, then return the cached handle — the second caller's closure must never run
- keep the given struct and signatures unchanged

**See**: §8.1 and §10.1 of the example notebook.

**Hint**: the cache is `OnceCell<Rc<Brick>>`: `get_or_init(|| ..)` hands back `&Rc<Brick>`, `.clone()` it for the returned handle. The `Weak` read is `upgrade()` — the `Option` *is* the safety.


In [ ]:
// (Everything here is already imported: RefCell, Rc, OnceCell, Weak.)

struct Brick {
    name: &'static str,
    parent: RefCell<Weak<Brick>>,
}
impl Brick {
    fn new(name: &'static str) -> Rc<Brick> {
        Rc::new(Brick { name, parent: RefCell::new(Weak::new()) })
    }
    /// Record `parent` as a Weak back-edge.
    ///
    /// TODO: replace `todo!()` with the downgrade.
    fn link(self: &Rc<Brick>, parent: &Rc<Brick>) {
        todo!()
    }
    /// Read the parent's name through the Weak; None for an orphan.
    ///
    /// TODO: replace `todo!()` with the upgrade + map.
    fn parent_name(self: &Rc<Brick>) -> Option<String> {
        todo!()
    }
}

/// Initialise the cache with the first caller's node; later callers read it.
///
/// TODO: replace `todo!()` with the get_or_init + clone.
fn build_cache(cache: &OnceCell<Rc<Brick>>, seed: &Rc<Brick>) -> Rc<Brick> {
    todo!()
}

// --- check (do not modify) ---
let foundation = Brick::new("foundation");
let course = Brick::new("course");
course.link(&foundation);
assert_eq!(course.parent_name(), Some("foundation".to_string()));
assert_eq!(foundation.parent_name(), None);
let cache: OnceCell<Rc<Brick>> = OnceCell::new();
assert_eq!(build_cache(&cache, &foundation).name, "foundation");
assert_eq!(build_cache(&cache, &course).name, "foundation"); // closure skipped
assert!(Rc::ptr_eq(&build_cache(&cache, &course), &foundation));
println!("Exercise 14.2 passed");


### Exercise 14.3 — The destructor report

**Goal**: produce the drop-timing evidence chapter 14 kept promising — early `mem::drop`, LIFO at the scope, and the count-zero free — in one function that returns the transcript.

**Given**: the `Pebble` type is declared with its `Drop` printer, and the log it records into lives **outside** the pebbles — an `Rc<Cell<Vec<&'static str>>>`, the §5 + §4 stack, so the evidence outlives the dropped pebbles.

**Objectives**

- implement `Pebble::stamp` to add this pebble's label to the shared log at drop time — `take` + push + `set`, the §5.1 idiom, because `Drop` only has `&mut self`
- implement `drop_report` to build three pebbles over one shared log, `mem::drop` the middle, and let the scope close over the other two — then read the transcript *after* the scope, in drop order
- return the transcript — early drop first, then LIFO — and let the check confirm it
- keep the given struct and signatures unchanged

**See**: §3.1, §5.1 and §10.1 of the example notebook.

**Hint**: an inner scope gives you the LIFO close; `mem::drop` fires exactly once where you call it; and because the log is an `Rc<Cell<..>>` clone shared with each pebble, it survives every drop it records.


In [ ]:
// (Cell is already imported; mem::drop joins the notebook here.)
use std::mem::drop;

struct Pebble {
    label: &'static str,
    log: Rc<Cell<Vec<&'static str>>>, // the recorder OUTLIVES every pebble
}
impl Pebble {
    fn new(label: &'static str, log: &Rc<Cell<Vec<&'static str>>>) -> Self {
        Pebble { label, log: Rc::clone(log) }
    }
    /// Record this pebble's drop into the shared log.
    ///
    /// TODO: replace `todo!()` with the take + push + set.
    fn stamp(&self) {
        todo!()
    }
}
impl Drop for Pebble {
    fn drop(&mut self) {
        self.stamp(); // the destructor's one allowed move: &mut self
    }
}

/// Drop the middle early; the other two close LIFO; return the transcript.
///
/// TODO: replace `todo!()` with the probe - one shared log, an inner scope,
/// mem::drop on the middle pebble, then read the log after the scope.
fn drop_report() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(drop_report(), "middle, top, bottom");
println!("Exercise 14.3 passed");


### What Comes Next

Chapter 14's receipts ledger is now yours to write without looking anything up. The track continues:

| Chapter | Title | What it settles |
|---|---|---|
| **15** | **Data Types and Their Libraries** | the crate as the unit you *consume*: `chrono`/`time`, `polars`, `uuid`, `rust_decimal` |
| 16 | Concurrency and Async/Await | the §9 wall comes down: `Send`/`Sync` in full, `Mutex`, channels, `tokio` |
| 17 | Capstone: Applying the Track | every receipt cashed at once — a real project wired end to end |

**Carry forward.** The eleven moves this notebook drilled: heap-indirection behind `Box` (and the recursion wall it fixes), hand-rolled `Deref`/`DerefMut` with the coercion ladder, destructor timing under `Drop`, counted ownership with `Rc`, copy-in/copy-out mutation with `Cell`, runtime-checked loans with `RefCell`, the `Rc<RefCell<T>>` stack, write-once cells, the `Arc` thread boundary, `Weak` back-edges that break cycles — and the seven E-codes that defend the whole system.
